# Main figures
Manufacturing-vintage footprints and deployment-associated carbon balances. Execute from the project root or code directory.

In [ ]:
# Setup
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import FuncNorm, LinearSegmentedColormap, TwoSlopeNorm, ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.collections import LineCollection
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.transforms import Bbox
ROOT=Path.cwd() if (Path.cwd()/'code').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'code'))
from audit_panel_alignment import require_matplotlib_panel_alignment
D=ROOT/'output/Main_Figure_Data'; O=ROOT/'output/Main_Figures'; Q=ROOT/'output/checks/main_figures'; S=ROOT/'output/Source_Data/CSV'
for p in [O,Q,S]:p.mkdir(parents=True,exist_ok=True)
mpl.rcParams.update({'font.family':['Arial','DejaVu Sans'],'font.sans-serif':['Arial','DejaVu Sans'],'font.size':7,
    'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,'legend.fontsize':6,
    'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
    'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white','legend.frameon':False})
INK='#233F62';POS='#0072B2';NEG='#D55E00';GREY='#87949C';TEAL='#299B8F'
COLORS={'Low':'#D89000','Reference':'#0072B2','High':'#AA3377'}
SCENARIO_MARKERS={'Low':'s','Reference':'o','High':'^'}
TABLES={}
def read(n):return pd.read_csv(D/(n+'.csv'))
def save_data(n,f):
    assert not f.empty and not f.columns.duplicated().any()
    TABLES[n]=f.reset_index(drop=True);TABLES[n].to_csv(S/(n+'.csv'),index=False)
def name(s):return {'uk':'UK','usa':'USA','row':'Rest of world'}.get(s,s.replace('_',' ').title())
def panel(ax,letter):
    ax.set_label(letter)
    ax.annotate(letter,(0,1),xycoords='axes fraction',xytext=(-6,9),textcoords='offset points',fontsize=9,fontweight='bold',color=INK,annotation_clip=False)
def export(fig,n,axes=None,**kwargs):
    fig.canvas.draw()
    if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
    require_matplotlib_panel_alignment(fig,json_out=str(Q/f'Figure_{n}.alignment.json'),strict=True,tolerance_pt=1.5,**kwargs)
    bbox=Bbox.from_extents(0,8/25.4,fig.get_figwidth(),fig.get_figheight()-8/25.4) if n==4 else Bbox.from_extents(0,7/25.4,fig.get_figwidth(),fig.get_figheight()) if n==5 else None
    fig.savefig(O/f'Figure_{n}.pdf',dpi=600,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.svg',dpi=600,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.png',dpi=300,bbox_inches=bbox)
    fig.savefig(O/f'Figure_{n}.tiff',dpi=600,bbox_inches=bbox,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig);print('Exported Figure',n,flush=True)
def show(n):
    if 'get_ipython' in globals():
        from IPython.display import display,Image
        display(Image(filename=str(O/f'Figure_{n}.png')))
ep=read('country_endpoints');high=ep.query('final_coverage_pct==75').set_index('country_tag')
negative=high.query('cumulative_net_gt<0').sort_values('cumulative_net_gt');tags=negative.index.tolist()
assert len(tags)==6



In [ ]:
# Figure 1
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
cartopy.config['data_dir']=ROOT/'input/maps'
grid=read('grid_reference');eligible=grid.query('net_gt>=0 and gross_gt>0');bad=grid.query('net_gt<0')
assert len(grid)==11047
lon=np.arange(-180,181);lat=np.arange(-90,91)
def raster(df,col):
    z=np.full((180,360),np.nan);ix=np.floor(df.lon_center+180).astype(int);iy=np.floor(df.lat_center+90).astype(int)
    ok=(ix>=0)&(ix<360)&(iy>=0)&(iy<180);z[iy[ok],ix[ok]]=df.loc[ok,col];return z
vmin=abs(grid.net_gt.min()*1000);vmax=grid.net_gt.max()*1000;soft=.05
def forward(x):
    x=np.asarray(x);return np.where(x<0,.5-.5*np.log1p(np.maximum(-x,0)/soft)/np.log1p(vmin/soft),.5+.5*np.log1p(np.maximum(x,0)/soft)/np.log1p(vmax/soft))
def inverse(y):
    y=np.asarray(y);return np.where(y<.5,-soft*np.expm1((1-2*y)*np.log1p(vmin/soft)),soft*np.expm1((2*y-1)*np.log1p(vmax/soft)))
netnorm=FuncNorm((forward,inverse),vmin=-vmin,vmax=vmax)
netmap=LinearSegmentedColormap.from_list('balance',[(0,'#8C2D04'),(.25,'#E66A12'),(.49,'#FEE6CE'),(.50,'#F7F7F7'),(.51,'#DCEAF4'),(.75,'#3B91BF'),(1,'#08446C')])
offsetmap=LinearSegmentedColormap.from_list('offset',['#FFF2CC','#F5C65D','#DE9315','#B2600B','#763B08'])
fig=plt.figure(figsize=(7.2047244,120/25.4));gs=fig.add_gridspec(2,2,left=.02,right=.975,bottom=.10,top=.91,wspace=.48,hspace=.58,width_ratios=[1.65,1])
a=fig.add_subplot(gs[0,0],projection=ccrs.Robinson(central_longitude=10));b=fig.add_subplot(gs[0,1]);c=fig.add_subplot(gs[1,0],projection=ccrs.Robinson(central_longitude=10));d=fig.add_subplot(gs[1,1])
for ax in [a,c]:
    ax.add_feature(cfeature.LAND.with_scale('110m'),facecolor='#ECEEEF',edgecolor='none',zorder=1)
    ax.add_feature(cfeature.OCEAN.with_scale('110m'),facecolor='#F5F8FA',edgecolor='none',zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale('110m'),edgecolor='#BCC3C6',linewidth=.25,zorder=4)
    ax.add_feature(cfeature.BORDERS.with_scale('110m'),edgecolor='#BCC3C6',linewidth=.15,zorder=4)
    ax.set_global()
    ax.set_xlim(-14172757.860816741,16060863.567199467)
    ax.set_ylim(ax.projection.transform_point(0,-55,ccrs.PlateCarree())[1],ax.projection.transform_point(0,75,ccrs.PlateCarree())[1])
    for lo,hi,bot,top in [(-25,33,43,71),(-63,-53,-28,-18)]:
        xx=np.r_[np.linspace(lo,hi,30),np.full(30,hi),np.linspace(hi,lo,30),np.full(30,lo)]
        yy=np.r_[np.full(30,bot),np.linspace(bot,top,30),np.full(30,top),np.linspace(top,bot,30)]
        ax.plot(xx,yy,color=INK,lw=.5,transform=ccrs.PlateCarree(),zorder=6)
mesh=a.pcolormesh(lon,lat,raster(grid,'net_gt')*1000,cmap=netmap,norm=netnorm,transform=ccrs.PlateCarree(),rasterized=True,zorder=3)
cb=fig.colorbar(mesh,ax=a,fraction=.024,pad=.035);ticks=[v for v in [-10,-1,-.1,0,.1,1,10,100,500] if -vmin<=v<=vmax]
cb.set_ticks(ticks);cb.set_ticklabels([f'{v:g}' for v in ticks]);cb.ax.tick_params(labelsize=5.5,length=2);cb.set_label('Net balance (Mt CO₂e)',fontsize=6)
mesh=c.pcolormesh(lon,lat,np.clip(raster(eligible,'offset_pct'),0,100),cmap=offsetmap,norm=TwoSlopeNorm(vmin=0,vcenter=20,vmax=80),transform=ccrs.PlateCarree(),rasterized=True,zorder=3)
c.pcolormesh(lon,lat,np.where(np.isfinite(raster(bad,'net_gt')),1.,np.nan),cmap=ListedColormap(['#7040A0']),vmin=0,vmax=1,transform=ccrs.PlateCarree(),rasterized=True,zorder=4)
cb=fig.colorbar(mesh,ax=c,fraction=.024,pad=.035,extend='max');cb.set_ticks([0,20,40,60,80]);cb.ax.tick_params(labelsize=5.5,length=2);cb.set_label('Gross savings offset (%)',fontsize=6)
c.legend(handles=[Patch(facecolor='#7040A0',label='Net-emitting cell')],loc='upper center',bbox_to_anchor=(.52,-.035),fontsize=6,handlelength=1.1)
vals=-negative.cumulative_net_gt*1000;b.barh(range(6),vals,color=NEG,height=.57)
b.set(yticks=range(6),yticklabels=[name(t) for t in tags],ylim=(5.6,-.6),xlim=(0,32),xticks=[0,10,20,30],xlabel='Cumulative net emissions (Mt CO₂e)')
for j,v in enumerate(vals):b.text(v+.5,j,f'{v:.1f}',va='center',fontsize=6)
zones=grid.groupby('climate_zone').era5_t2m_c.median().to_frame('temperature_c');zones['offset_pct']=eligible.groupby('climate_zone').offset_pct.median();zones=zones.sort_values('temperature_c')
short={'humid_continental_temperate':'Humid temperate','humid_subtropical_dense':'Humid subtropical','tropical_wet_dense':'Tropical wet','hot_arid_sparse':'Hot arid sparse','hot_arid_dense_riverine':'Hot arid riverine','hot_arid_low_density':'Hot arid low density'}
d.barh(range(6),zones.offset_pct,color='#D89000',height=.57);d.set(yticks=range(6),yticklabels=[short[z] for z in zones.index],ylim=(5.6,-.6),xlim=(0,zones.offset_pct.max()*1.5),xlabel='Median gross savings offset (%)')
for j,row in enumerate(zones.itertuples()):
    d.text(row.offset_pct+.12,j,f'{row.offset_pct:.1f}',va='center',fontsize=6)
    d.text(.985,j,f'{row.temperature_c:.1f}',transform=d.get_yaxis_transform(),va='center',ha='right',fontsize=5.7)
d.text(.985,1.04,'°C',transform=d.transAxes,ha='right',fontsize=6)
for ax,l in zip([a,b,c,d],'abcd'):panel(ax,l)
export(fig,1,[a,b,c,d],exemptions=[{'panels':['a','c'],'checks':['row','panel-label'],'reason':'Robinson projection fixes map aspect; adjacent bars use full quantitative plot height.'}])
projected=a.projection.transform_points(ccrs.PlateCarree(),grid.lon_center.to_numpy(),grid.lat_center.to_numpy())
visible=(np.floor(grid.lat_center)>=-55)&(np.floor(grid.lat_center)<75)&(projected[:,0]>=-14172757.860816741)&(projected[:,0]<=16060863.567199467)
f=grid.loc[visible,['Id','lon_center','lat_center']].copy();f['net_balance_mt_co2e']=grid.loc[visible,'net_gt']*1000;save_data('Figure_1a',f)
save_data('Figure_1b',pd.DataFrame({'country':tags,'net_emissions_mt_co2e':vals.to_numpy()}))
f=grid.loc[visible&((grid.net_gt<0)|((grid.net_gt>=0)&(grid.gross_gt>0))),['Id','lon_center','lat_center']].copy();f['series']=np.where(grid.loc[f.index,'net_gt']<0,'Net-emitting cell','Gross savings offset');f['offset_pct']=np.where(grid.loc[f.index,'net_gt']<0,np.nan,np.clip(grid.loc[f.index,'offset_pct'],0,100));save_data('Figure_1c',f)
save_data('Figure_1d',zones.reset_index());show(1)



In [ ]:
# Figure 2
countries=tags.copy();budget=high.loc[countries];den=budget.cumulative_generation_twh*1e-6
comps=[('displaced_grid','Grid credit',1,'#009E73'),('pv_embodied','PV system emissions',-1,'#B8BDC5'),('battery','Battery emissions',-1,'#0072B2'),('cooling','Cooling electricity',-1,'#E69F00'),('ac','AC equipment',-1,'#AA3377'),('roof_credit','Roof credit',1,'#F0E442')]
values=pd.DataFrame({'country':countries})
for k,lab,sign,col in comps:values[k+'_g_kwh']=(sign*budget['cumulative_'+k+'_gt']/den).to_numpy()
values['gross_savings_g_kwh']=values.displaced_grid_g_kwh+values.pv_embodied_g_kwh
values['net_g_kwh']=(budget.cumulative_net_gt/den).to_numpy()
np.testing.assert_allclose(values[[k+'_g_kwh' for k,_,_,_ in comps]].sum(1),values.net_g_kwh,atol=1e-9)
fig,a=plt.subplots(figsize=(7.2047244,3.1));fig.subplots_adjust(left=.18,right=.95,bottom=.18,top=.81)
pos=np.zeros(6);neg=np.zeros(6)
for k,lab,sign,col in comps:
    v=values[k+'_g_kwh'].to_numpy();a.barh(range(6),v,left=np.where(v>=0,pos,neg),color=col,height=.47,edgecolor='white',linewidth=.25);pos+=np.maximum(v,0);neg+=np.minimum(v,0)
for j,row in values.iterrows():
    width=-(row.battery_g_kwh+row.cooling_g_kwh+row.ac_g_kwh);right=row.pv_embodied_g_kwh
    a.add_patch(Rectangle((right-width,j-.235),width,.47,facecolor='none',edgecolor='black',lw=.8,zorder=6))
# Paired accounting balances share the same physical generation and delivery.
for j,row in values.iterrows():
    a.plot([row.net_g_kwh,row.gross_savings_g_kwh],[j,j],color='white',lw=2.0,zorder=7)
    a.plot([row.net_g_kwh,row.gross_savings_g_kwh],[j,j],color='#171717',lw=.65,zorder=8)
a.scatter(values.gross_savings_g_kwh,range(6),marker='o',s=22,facecolors='white',edgecolors='#171717',lw=.8,zorder=10)
a.scatter(values.net_g_kwh,range(6),marker='D',s=22,c='#171717',edgecolors='white',lw=.65,zorder=10)
a.axvline(0,c=GREY,lw=.6);a.set(yticks=range(6),yticklabels=[name(t) for t in countries],ylim=(5.6,-.6),xlim=(np.floor(neg.min()*1.12/10)*10,np.ceil(pos.max()*1.12/10)*10),xlabel='Cumulative contribution\n(g CO₂e per generated kWh)')
handles=[Patch(color=col,label=lab) for _,lab,_,col in comps]+[Line2D([],[],ls='none',marker='o',mfc='white',mec='#171717',ms=4,label='Grid credit − PV emissions'),Line2D([],[],ls='none',marker='D',mfc='#171717',mec='white',ms=4,label='Net balance'),Patch(facecolor='none',edgecolor='black',label='Additional emissions')]
fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.56,.99),ncol=3,fontsize=5.8,columnspacing=1.0,labelspacing=.45,handlelength=1.1)
export(fig,2,[a]);save_data('Figure_2',values);show(2)


In [ ]:
# Figure 3
shown=['sweden','france','germany'];years=[2030,2050]
annual=read('annual_reference').query('final_coverage_pct==75 and country_tag in @shown and year in @years').copy()
fig=plt.figure(figsize=(7.2047244,5.1));top=fig.add_gridspec(1,3,left=.13,right=.97,bottom=.535,top=.835,wspace=.32);axes=[];points=[]
colors={'balance':'#454545','increase':'#0072B2','decrease':'#D55E00'}
for i,tag in enumerate(shown):
    ax=fig.add_subplot(top[0,i]);ax.set_label('a'+str(i));axes.append(ax)
    z=annual.query('country_tag==@tag').set_index('year').loc[years];den=z.annual_generation_twh*1e-6
    v={key:sign*z['annual_'+key+'_gt']/den for key,sign in [('displaced_grid',1),('pv_embodied',-1),('battery',-1),('cooling',-1),('ac',-1),('roof_credit',1)]}
    net=z.annual_net_gt/den
    np.testing.assert_allclose(sum(v.values()).to_numpy(),net.to_numpy(),atol=1e-9)
    change={key:float(val.loc[2050]-val.loc[2030]) for key,val in v.items()}
    steps=[('2030','balance',float(net.loc[2030])),('Grid','change',change['displaced_grid']),('PV','change',change['pv_embodied']),('Battery','change',change['battery']),('Other','change',sum(change[k] for k in ['cooling','ac','roof_credit'])),('2050','balance',float(net.loc[2050]))]
    running=steps[0][2]
    for j,(term,kind,value) in enumerate(steps):
        start=0 if kind=='balance' else running;end=value if kind=='balance' else running+value
        color=colors['balance'] if kind=='balance' else colors['increase' if value>=0 else 'decrease']
        width=.72 if kind=='balance' else .52
        ax.bar(j,end-start,bottom=start,width=width,color=color,edgecolor='white',linewidth=.35,zorder=3)
        if kind=='change':running=end
        if j<5:ax.plot([j+width/2,j+1-(.36 if j==4 else .26)],[end,end],c=GREY,lw=.55,ls=':',zorder=2)
        anchor=max(start,end) if value>=0 else min(start,end)
        above=value>=0 or (tag=='germany' and term=='Battery')
        if above:anchor=max(start,end)
        ax.annotate(f'{value:+.1f}' if kind=='change' else f'{value:.1f}',(j,anchor),xytext=(0,4 if above else -4),textcoords='offset points',ha='center',va='bottom' if above else 'top',fontsize=5.2,fontweight='bold' if kind=='balance' else 'normal',color=color)
        points.append(dict(country=tag,step=term,role=kind,value_g_kwh=value,start_g_kwh=start,end_g_kwh=end))
    np.testing.assert_allclose(running,net.loc[2050],atol=1e-9)
    ax.axhline(0,c=GREY,lw=.6);ax.set(xticks=range(6),xticklabels=['2030\nbalance','Grid\ncredit','PV\nsystem','Battery','Other','2050\nbalance'],xlim=(-.65,5.65))
    ax.set_ylim({'sweden':(-43,4),'france':(-20,26),'germany':(-12,265)}[tag]);ax.tick_params(axis='x',labelsize=5.3,pad=3)
    for tick in [ax.get_xticklabels()[0],ax.get_xticklabels()[-1]]:tick.set_fontweight('bold')
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4));ax.set_title(name(tag),loc='left',fontsize=6.5,pad=5)
panel(axes[0],'a');axes[0].set_label('a0');axes[0].texts[-1].set_position((-6,19))
fig.text(.038,.68,'Annual net mitigation\n(g CO₂e per generated kWh)',rotation=90,va='center',ha='center',fontsize=7)
fig.legend(handles=[Patch(color=colors['balance'],label='Annual net balance'),Patch(color=colors['increase'],label='Increase in net mitigation'),Patch(color=colors['decrease'],label='Decrease in net mitigation')],loc='upper center',bbox_to_anchor=(.56,.925),ncol=3,fontsize=5.8,columnspacing=1.6,handlelength=1.1)
bottom=fig.add_gridspec(1,3,left=.13,right=.97,bottom=.075,top=.375,wspace=.32);time=read('manufacturing_time_paths');line_data=[];bottom_axes=[]
series=[('Evolving manufacturing grid','#0072B2','-','o'),('Frozen 2025 manufacturing grid','#454545','--','s')]
for i,tag in enumerate(['sweden','france','germany']):
    ax=fig.add_subplot(bottom[0,i]);ax.set_label('b'+str(i));axes.append(ax);bottom_axes.append(ax)
    curves={}
    for label,color,ls,marker in series:
        z=time.query('country_tag==@tag and series==@label').sort_values('year');curves[label]=z
        ax.plot(z.year,z.cumulative_net_gt*1000,c=color,ls=ls,lw=1.1)
        ax.scatter([2050],[z.cumulative_net_gt.iloc[-1]*1000],s=15,marker=marker,facecolors=color if ls=='-' else 'white',edgecolors=color,linewidths=.75,zorder=4)
        line_data.extend(dict(country=tag,year=int(y),series=label,net_mitigation_mt_co2e=v*1000) for y,v in zip(z.year,z.cumulative_net_gt))
    baseline=curves[series[0][0]];counter=curves[series[1][0]]

    ax.axhline(0,c=GREY,lw=.6,ls=':');ax.set(xlim=(2024,2059.5),xticks=[2025,2030,2040,2050],xlabel='Year');ax.yaxis.set_major_locator(MaxNLocator(nbins=4));ax.set_title(name(tag),loc='left',fontsize=6.5,pad=5);ax.tick_params(axis='x',labelsize=5.7);ax.margins(y=.13)
    np.testing.assert_array_equal(baseline.year.to_numpy(),counter.year.to_numpy())
    evolving=baseline.cumulative_net_gt.to_numpy()*1000;frozen=counter.cumulative_net_gt.to_numpy()*1000
    gain=float(evolving[-1]-frozen[-1])
    ax.fill_between(baseline.year.to_numpy(),frozen,evolving,color='#0072B2',alpha=.12,lw=0,zorder=1)
    xgap=2051.1;endpoint_low,endpoint_high=frozen[-1],evolving[-1]
    ax.plot([2050,xgap],[endpoint_low,endpoint_low],c='#0072B2',lw=.55)
    ax.plot([2050,xgap],[endpoint_high,endpoint_high],c='#0072B2',lw=.55)
    ax.plot([xgap,xgap],[endpoint_low,endpoint_high],c='#0072B2',lw=.75)
    ax.annotate(f'+{gain:.2f}\nMt',xy=(xgap,(endpoint_low+endpoint_high)/2),xytext=(4,0),textcoords='offset points',ha='left',va='center',fontsize=5.6,c='#0072B2')
    for row in line_data:
        if row['country']==tag:
            row['manufacturing_grid_gain_2050_mt_co2e']=gain if row['year']==2050 and row['series']==series[0][0] else None
    if tag in ['sweden','germany']:
        lo,hi=(2025,2028) if tag=='sweden' else (2048,2050)
        ins=ax.inset_axes([.18,.12,.38,.30] if tag=='sweden' else [.12,.70,.31,.25]);ins.set_label('inset_'+tag)
        for label,color,ls,marker in series:
            zz=curves[label].query('@lo<=year<=@hi')
            ins.plot(zz.year,zz.cumulative_net_gt*1000,c=color,ls=ls,lw=.85,marker=marker,ms=2.4,mfc=color if ls=='-' else 'white',mec=color,mew=.5)
        inz=baseline.query('@lo<=year<=@hi');inc=counter.query('@lo<=year<=@hi')
        ins.fill_between(inz.year.to_numpy(),inc.cumulative_net_gt.to_numpy()*1000,inz.cumulative_net_gt.to_numpy()*1000,color='#0072B2',alpha=.12,lw=0,zorder=1)
        if tag=='sweden':ins.axhline(0,c=GREY,lw=.5,ls=':')
        ins.set(xticks=[lo,hi]);ins.margins(y=.18);ins.yaxis.set_major_locator(MaxNLocator(nbins=3));ins.tick_params(labelsize=5,pad=1,length=2)
        if tag=='germany':
            ins.set_xlim(2047.85,2050.45)
            ins.plot([2050,2050.18,2050.18,2050],[endpoint_low,endpoint_low,endpoint_high,endpoint_high],c='#0072B2',lw=.65)
        for spine in ins.spines.values():spine.set_visible(True);spine.set_color('#999999');spine.set_linewidth(.5)
panel(bottom_axes[0],'b');bottom_axes[0].set_label('b0');bottom_axes[0].texts[-1].set_position((-6,18))
fig.text(.045,.225,'Cumulative net mitigation (Mt CO₂e)',rotation=90,va='center',fontsize=7)
fig.legend(handles=[Line2D([],[],c=color,ls=ls,marker=mark,mfc=color if ls=='-' else 'white',ms=3,label=label) for label,color,ls,mark in series]+[Patch(facecolor='#0072B2',alpha=.12,edgecolor='none',label='Gain from manufacturing-grid decarbonisation')],loc='center',bbox_to_anchor=(.56,.445),ncol=3,fontsize=5.7,columnspacing=1.2,handlelength=1.8)
export(fig,3,axes,column_groups=[{'id':f'country-{i}','panels':[f'a{i}',f'b{i}']} for i in range(3)]);save_data('Figure_3a',pd.DataFrame(points));save_data('Figure_3b',pd.DataFrame(line_data));show(3)


In [ ]:
# Figure 4
lever=read('interventions');lever=pd.concat([lever.iloc[:1],lever.iloc[1:].sort_values('reduction_pct',ascending=False)])
u=read('country_sensitivity_intensity')
rows=[]
for tag,z in u.groupby('country_tag'):
    q=np.quantile(z.net_g_co2e_kwh,[.25,.5,.75]);rows.append(dict(country=tag,p25=q[0],median=q[1],p75=q[2],negative_scenarios_pct=100*(z.net_gt<0).mean(),baseline=high.loc[tag,'cumulative_net_gt']/high.loc[tag,'cumulative_generation_twh']*1e6))
summary=pd.DataFrame(rows).set_index('country').loc[tags].reset_index()
fig,(a,b)=plt.subplots(1,2,figsize=(7.2047244,109/25.4));fig.subplots_adjust(left=.21,right=.94,bottom=.20,top=.86,wspace=.59)
labels={'Coverage 75 to 25%':'Coverage 75 → 25%','Low-emission manufacturing':'Low-emission\nmanufacturing','Battery footprint halved':'Battery footprint\nhalved','Direct use +20 pp':'Direct use +20 pp','Roof credit doubled':'Roof credit ×2','Battery life +50%':'Battery life +50%','Cooling demand halved':'Cooling demand\nhalved'}
for j,row in enumerate(lever.itertuples()):
    a.barh(j,row.reduction_pct,height=.5,color='#707780' if j==0 else '#0072B2',hatch='///' if j==0 else None,edgecolor='white')
    a.text(row.reduction_pct+1,j,f'{row.reduction_pct:.1f}',va='center',fontsize=6)
a.set(yticks=range(len(lever)),yticklabels=[labels[v] for v in lever.intervention],ylim=(len(lever)-.4,-.6),xlim=(0,105),xticks=[0,25,50,75,100],xlabel='Reduction in cumulative\nnet emissions (%)');a.spines['left'].set_visible(False);a.tick_params(axis='y',length=0)
for j,row in enumerate(summary.itertuples()):
    vals=u.query('country_tag==@row.country').net_g_co2e_kwh.to_numpy();assert len(vals)==2048
    col='#0072B2'
    vi=b.violinplot([vals],positions=[j],orientation='horizontal',widths=.7,showextrema=False)
    for body in vi['bodies']:body.set_facecolor('#C6DEEF');body.set_edgecolor(col);body.set_linewidth(.65);body.set_alpha(1)
    b.plot([row.p25,row.p75],[j,j],c=col,lw=3);b.scatter(row.median,j,s=17,c='white',edgecolors=col,lw=.7,zorder=4);b.scatter(row.baseline,j,marker='D',s=20,c='#171717',edgecolors='white',lw=.6,zorder=5)
    b.text(1.03,j,f'{row.negative_scenarios_pct:.1f}',transform=b.get_yaxis_transform(),va='center',fontsize=6)
b.set(yticks=range(6),yticklabels=[name(t) for t in summary.country],ylim=(5.6,-.6),xlabel='Net balance\n(g CO₂e per generated kWh)');b.axvline(0,c=GREY,ls=':',lw=.65)
b.text(1.055,1.04,'Net-emission\nscenarios (%)',transform=b.transAxes,fontsize=5.7,ha='center')
for ax,l in zip([a,b],'ab'):panel(ax,l)
export(fig,4);save_data('Figure_4a',lever[['intervention','reduction_pct']]);save_data('Figure_4b_values',u[['sample_id','country_tag','net_g_co2e_kwh']]);save_data('Figure_4b_summary',summary);show(4)



In [ ]:
# Figure 5
all_results=read('matched_scenarios');base=all_results.query('sample_id==0');data=all_results.query('sample_id>0');assert len(data)==2048*6
uniform_ref=data.query('scenario=="Reference" and strategy=="Uniform"').set_index('sample_id').net_gt
base_ref=base.query('scenario=="Reference" and strategy=="Uniform"').net_gt.iloc[0]
cats=[('Screened','Reference','Screened','#171717'),('Low','Low','Uniform',COLORS['Low']),('Reference','Reference','Uniform',COLORS['Reference']),('High','High','Uniform',COLORS['High']),('Combined','Low','Screened','#009E73')]
paired=pd.DataFrame({lab:data.query('scenario==@sc and strategy==@st').set_index('sample_id').net_gt-uniform_ref for lab,sc,st,col in cats}).sort_index();assert paired.notna().all().all()
fig,axes=plt.subplots(2,2,figsize=(7.2047244,6.0));fig.subplots_adjust(left=.115,right=.98,bottom=.17,top=.94,hspace=.34,wspace=.48);a,b,c,d=axes.flat
segments=np.stack([np.tile(np.arange(5),(len(paired),1)),paired.to_numpy()],axis=2);a.add_collection(LineCollection(segments,colors=GREY,alpha=.006,linewidths=.25,rasterized=True))
qrows=[]
for j,(lab,sc,st,col) in enumerate(cats):
    q=np.quantile(paired[lab],[.05,.5,.95]);bv=base.query('scenario==@sc and strategy==@st').net_gt.iloc[0]-base_ref
    a.errorbar(j,q[1],yerr=[[q[1]-q[0]],[q[2]-q[1]]],fmt=['D','s','o','^','P'][j],color=col,ms=5,mec='white',mew=.45,capsize=3,elinewidth=1.5,zorder=4)
    qrows.append(dict(category=lab,p05_gt=q[0],median_gt=q[1],p95_gt=q[2]))
a.autoscale_view();a.axhline(0,c=GREY,lw=.6,ls=':');a.set(xlim=(-.45,4.55),xticks=range(5),xticklabels=[r[0] for r in cats],ylabel='Gain vs reference uniform (Gt CO₂e)');a.tick_params(axis='x',labelsize=5.7)
rank=read('ranking_comparisons')
ranking_draws=read('ranking_scenarios').query('sample_id>0');ranking_summary=[]
for j,row in enumerate(rank.itertuples()):
    z=ranking_draws.query('rule==@row.rule');v=z.full_minus_alternative_gt;assert len(v)==2048
    jitter=((z.sample_id.to_numpy()*37)%101/100-.5)*.27
    b.scatter(v,j+jitter,s=3,c=GREY,alpha=.07,edgecolors='none',rasterized=True)
    q=np.quantile(v,[.05,.5,.95]);tie=np.max(abs(v))<1e-8
    b.errorbar(q[1],j,xerr=[[q[1]-q[0]],[q[2]-q[1]]],fmt='o',color=INK if tie else '#0072B2',mfc='white' if tie else '#0072B2',ms=5,capsize=3,elinewidth=1.5,zorder=4)
    ranking_summary.append(dict(rule=row.rule,retained_regions=row.retained_regions,p05_gt=q[0],median_gt=q[1],p95_gt=q[2]))
b.axvline(0,c=GREY,lw=.65,ls=':');b.set(yticks=range(6),yticklabels=[f'{r.rule} ({r.retained_regions})' for r in rank.itertuples()],ylim=(5.65,-.65),xlabel='Screened − alternative (Gt CO₂e)');b.tick_params(axis='y',length=0,labelsize=5.8);b.spines['left'].set_visible(False)
bins=np.linspace(.5,1.5,9);binrows=[];pointrows=[];cdf=[]
for idx,sc in enumerate(['Low','Reference','High']):
    pair=data.query('scenario==@sc').pivot(index='sample_id',columns='strategy',values='net_gt');gain=pair.Screened-pair.Uniform
    ef=data.query('scenario==@sc and strategy=="Uniform"').set_index('sample_id').loc[gain.index,'displaced_ef_multiplier']
    c.scatter(ef,gain,s=3,c=COLORS[sc],alpha=.05,edgecolors='none',rasterized=True)
    pointrows.extend(dict(sample_id=int(sid),scenario=sc,displaced_ef_multiplier=float(xx),screening_gain_gt=float(yy)) for sid,xx,yy in zip(gain.index,ef,gain))
    rr=[]
    for k,(lo,hi) in enumerate(zip(bins[:-1],bins[1:])):
        mask=(ef>=lo)&(ef<hi if k<7 else ef<=hi);q=np.quantile(gain[mask],[.05,.5,.95]);rr.append(dict(scenario=sc,bin_center=(lo+hi)/2,plot_x=(lo+hi)/2+(idx-1)*.014,p05_gt=q[0],median_gt=q[1],p95_gt=q[2]))
    ff=pd.DataFrame(rr);binrows.extend(rr);c.errorbar(ff.plot_x,ff.median_gt,yerr=[ff.median_gt-ff.p05_gt,ff.p95_gt-ff.median_gt],fmt=['s-','o-','^-'][idx],c=COLORS[sc],ms=4,mfc='white',mec=COLORS[sc],mew=.75,lw=1,capsize=1.4,elinewidth=.75,label=sc)
    for st,ls in [('Uniform','--'),('Screened','-')]:
        cnt=data.query('scenario==@sc and strategy==@st').negative_countries.to_numpy();xx=np.arange(int(data.negative_countries.max())+1);yy=np.array([(cnt<=v).mean()*100 for v in xx]);d.step(xx,yy,where='post',c=COLORS[sc],ls=ls,lw=1.1,marker=SCENARIO_MARKERS[sc],ms=[4.3,3.5,3.0][idx],mfc='white' if st=='Uniform' else COLORS[sc],mec=COLORS[sc],mew=.65)
        cdf.extend(dict(scenario=sc,strategy=st,negative_countries=int(v),cumulative_scenarios_pct=float(w)) for v,w in zip(xx,yy))
c.axhline(0,c=GREY,lw=.6,ls=':');c.set(xlim=(.5,1.5),xticks=[.5,.75,1,1.25,1.5],xlabel='Displaced-EF multiplier',ylabel='Screened − uniform (Gt CO₂e)')
mx=int(data.negative_countries.max());d.set(xlim=(-.35,mx+.35),ylim=(-3,104),xticks=np.arange(0,mx+1,2),yticks=[0,25,50,75,100],xlabel='Net-emitting countries',ylabel='Scenarios at or below (%)')
d.legend(handles=[Line2D([],[],c='#171717',ls='--',marker='o',ms=3,mfc='white',label='Uniform'),Line2D([],[],c='#171717',marker='o',ms=3,label='Screened')],loc='lower right',fontsize=5.7)
for ax,l in zip(axes.flat,'abcd'):panel(ax,l)
fig.legend(handles=[Line2D([],[],color=COLORS[s],lw=1.3,marker=SCENARIO_MARKERS[s],mfc='white',ms=4,label=s) for s in ['Low','Reference','High']],loc='lower center',bbox_to_anchor=(.54,.066),ncol=3,fontsize=6.5)
export(fig,5);save_data('Figure_5a_values',paired.rename_axis('sample_id').reset_index());save_data('Figure_5a_summary',pd.DataFrame(qrows));save_data('Figure_5b',ranking_draws[['sample_id','rule','full_minus_alternative_gt']]);save_data('Figure_5b_summary',pd.DataFrame(ranking_summary));save_data('Figure_5c_values',pd.DataFrame(pointrows));save_data('Figure_5c_summary',pd.DataFrame(binrows));save_data('Figure_5d',pd.DataFrame(cdf));show(5)



In [ ]:
# Source data
(Q/'source_data_manifest.json').write_text(json.dumps({n:dict(rows=len(f),columns=list(f)) for n,f in TABLES.items()},indent=2),encoding='utf8')
(Q/'source_data_values.json').write_text(json.dumps({n:[list(f.columns)]+f.astype(object).where(f.notna(),None).values.tolist() for n,f in TABLES.items()},ensure_ascii=False),encoding='utf8')
